# Chapter 2 - TRANSFORM

## 00 - IMPORT & EXPLOR DATA ANALYSIS

In [15]:
# 1.- IMPORT: Required libraries
import pandas as pd
from pathlib import Path

# 2.- DEFINE: Data folder
data_dir = Path("../data")

# 3.- LOAD: Cleaned dataset from Chapter 1
df = pd.read_csv(data_dir / "worldbank_long.csv")

# 4.- INSPECT: Verify data
print("Dataset shape:", df.shape)

display(df.head())

Dataset shape: (674, 6)


,country_code,country_name,indicator_code,indicator_name,year,value
0,BGR,Bulgaria,NY.GDP.PCAP.CD,gdp_per_capita,2025,20328.166707
1,BGR,Bulgaria,NY.GDP.PCAP.CD,gdp_per_capita,2024,17596.916762
2,BGR,Bulgaria,NY.GDP.PCAP.CD,gdp_per_capita,2023,15854.019289
3,BGR,Bulgaria,NY.GDP.PCAP.CD,gdp_per_capita,2022,13999.911336
4,BGR,Bulgaria,NY.GDP.PCAP.CD,gdp_per_capita,2021,12966.808520


## 01 - RESHAPE DATA (Convert from LONG to WIDE) 
Se reorganizan los datos, para que sean mas faciles de analizar, para reportes financieros y analisis estadisticos. 

### Step 1. Crear un DataFrame en formato LONG

En formato LONG, cada indicador ocupa una fila diferente.

```python
import pandas as pd

data = {
    "País": ["BGR", "BGR", "BGR"],
    "Año": [2024, 2024, 2024],
    "Indicador": ["GDP", "Inflación", "Desempleo"],
    "Valor": [17596, 2.45, 4.20]
}

df = pd.DataFrame(data)
display(df)
```

**Resultado:**

| País | Año | Indicador | Valor |
|---|---|---|---|
| BGR | 2024 | GDP | 17596 |
| BGR | 2024 | Inflación | 2.45 |
| BGR | 2024 | Desempleo | 4.20 |

### Step 2. Convertir de LONG a WIDE

Usamos `pivot_table()` para convertir los indicadores de filas a columnas.

```python
df_wide = df.pivot_table(
    index=["País", "Año"],
    columns="Indicador",
    values="Valor",
    aggfunc="mean"
).reset_index()

display(df_wide)
```

**Resultado:**

| País | Año | Desempleo | GDP | Inflación |
|---|---|---|---|---|
| BGR | 2024 | 4.20 | 17596 | 2.45 |

> **Nota:** Pandas normalmente ordena las columnas de los indicadores alfabéticamente (A–Z).

### Step 3. Convertir de WIDE a LONG

Usamos `melt()` para convertir las columnas de indicadores nuevamente en filas.

```python
df_long = df_wide.melt(
    id_vars=["País", "Año"],
    var_name="Indicador",
    value_name="Valor"
)

display(df_long)
```

**Resultado:**

| País | Año | Indicador | Valor |
|---|---|---|---|
| BGR | 2024 | Desempleo | 4.20 |
| BGR | 2024 | GDP | 17596 |
| BGR | 2024 | Inflación | 2.45 |

### To recall

| Método | ¿Qué hace? | Transformación |
|---|---|---|
| `pivot()` | Convierte filas en columnas sin agregar valores | LONG → WIDE |
| `pivot_table()` | Convierte filas en columnas y permite agregar valores | LONG → WIDE |
| `melt()` | Convierte columnas en filas | WIDE → LONG |
| `reset_index()` | Convierte el índice en columnas normales | Reorganiza el índice |

**Idea clave:**
- **LONG → WIDE:** los indicadores pasan de filas a columnas.
- **WIDE → LONG:** los indicadores pasan de columnas a filas.
- `pivot_table()` permite calcular promedios u otras agregaciones si hay registros repetidos.


In [12]:
# 1.- PIVOT THE DATA: from LONG FORMAT to WIDE FORMAT 
# Long: each row = one country/year/indicator/value
# Wide: each row = one country/year, columns = indicators

df_clean_wide = df.pivot_table(
    index= ['country_code', 'country_name', 'year'],
    columns= 'indicator_name', 
    values= 'value' 
).reset_index()

# 2.- REMOVE THE COLUMN NAME:  "indicator_name" that pandas adds
df_clean_wide.columns.name = None 


#3.- SORT VALUES (Los paises alfabeticamente por código ascending y los años decending).
df_clean_wide = df_clean_wide.sort_values(['country_code', 'year'], 
                              ascending=[True, False]
                             )

## 02 - VALIDATE TRANSFORMED DATA
#### 1. `df.shape` — Conocer las dimensiones del DataFrame

El atributo `shape` devuelve una tupla con dos valores:

`(número de filas, número de columnas)`

**Ejemplo:**

```python
print(df_wide.shape)
```

**Resultado hipotético:**

```text
(150, 5)
```

Esto significa que nuestro DataFrame contiene:

- **150 filas:** registros u observaciones.
- **5 columnas:** variables disponibles.

#### 2. `df.columns.tolist()` — Obtener los nombres de las columnas

La expresión `columns.tolist()` devuelve una lista de Python con los nombres de todas las columnas del DataFrame.

**Ejemplo:**

```python
print(df_wide.columns.tolist())
```

**Resultado hipotético:**

```python
['País', 'Año', 'GDP', 'Inflación', 'Desempleo']
``

In [11]:

print("="*50)
print("TRANSFORMED DATAFRAME (WIDE FORMAT)")
print("=" *50)
print(f"\nShape:{df_clean_wide.shape}")
print(f"\nColumns: {df_clean_wide.columns.tolist()}")
print(f"\nFirst 10 rows:")
df_clean_wide.head(10)


TRANSFORMED DATAFRAME (WIDE FORMAT)

Shape:(225, 6)

Columns: ['country_code', 'country_name', 'year', 'gdp_per_capita', 'inflation_rate', 'unemployment_rate']

First 10 rows:


,country_code,country_name,year,gdp_per_capita,inflation_rate,unemployment_rate
14,BGR,Bulgaria,2025,20328.166707,4.580844,3.536
13,BGR,Bulgaria,2024,17596.916762,2.446519,4.200
12,BGR,Bulgaria,2023,15854.019289,9.442841,4.319
11,BGR,Bulgaria,2022,13999.911336,15.325259,4.148
10,BGR,Bulgaria,2021,12966.808520,3.297744,5.166
9,BGR,Bulgaria,2020,10760.761618,1.672441,5.037
8,BGR,Bulgaria,2019,10354.250041,3.103729,4.148
7,BGR,Bulgaria,2018,9849.887700,2.814545,5.211
6,BGR,Bulgaria,2017,8697.133733,2.061596,6.164
5,BGR,Bulgaria,2016,7822.898864,-0.798650,7.575


## 03 - SAVE TRANSFORMED DATA  

In [13]:
from pathlib import Path

# 1.- DEFINE: Data folder
data_dir = Path("../data")
data_dir.mkdir(parents=True, exist_ok=True)

# 2.- SAVE: Wide format dataset
df_clean_wide.to_csv(
    data_dir / "worldbank_wide.csv",
    index=False                 #Save it without the Index column 
)

# 3.- VALIDATE: Confirm saved file
print("Chapter 2 completed!")
print(f"Records saved: {len(df_clean_wide)}")
print(f"Columns saved: {len(df_clean_wide.columns)}")
print("File saved: worldbank_wide.csv")

Chapter 2 completed!
Records saved: 225
Columns saved: 6
File saved: worldbank_wide.csv


- DataFrame: Está en la memoria de Python. Si cierras Jupyter o reinicias el kernel, perderás esa variable. Por eso, en un pipeline ETL, normalmente queremos guardar el resultado en un archivo como CSV o Parquet.